In [1]:

print("Week 2: Mini Semantic Search")
print("Loading, cleaning, embedding, and searching FAQs.")

Week 2: Mini Semantic Search
Loading, cleaning, embedding, and searching FAQs.


In [2]:
import pandas as pd

faqs = [
    {
        "question": "How do I reset my password?",
        "answer": "Click Forgot Password on the login page and follow the instructions sent to your email."
    },
    {
        "question": "How can I contact support?",
        "answer": "Email support@example.com and describe the problem you are experiencing."
    },
    {
        "question": "How do I cancel my subscription?",
        "answer": "Open account settings, select Manage Subscription, and click Cancel."
    },
    {
        "question": "How do I download an invoice?",
        "answer": "Open Billing, find your payment history, and select Download Invoice."
    },
    {
        "question": "Can I change my email address?",
        "answer": "Open Profile Settings, update your email address, and verify the new address."
    },
    {
        "question": "Why am I not receiving verification emails?",
        "answer": "Check your spam folder and confirm that your email address is correct."
    },
    {
        "question": "Which payment methods are accepted?",
        "answer": "Major credit and debit cards are accepted, depending on your country."
    },
    {
        "question": "How do I delete my account?",
        "answer": "Open account privacy settings and select Delete Account."
    }
]

df = pd.DataFrame(faqs)

df.to_csv("faqs.csv", index=False)

print("Dataset created successfully!")
print("Number of FAQs:", len(df))
df.head()

Dataset created successfully!
Number of FAQs: 8


,question,answer
0,How do I reset my password?,Click Forgot Password on the login page and fo...
1,How can I contact support?,Email support@example.com and describe the pro...
2,How do I cancel my subscription?,"Open account settings, select Manage Subscript..."
3,How do I download an invoice?,"Open Billing, find your payment history, and s..."
4,Can I change my email address?,"Open Profile Settings, update your email addre..."


In [3]:
import pandas as pd

df = pd.read_csv("faqs.csv")

print("Dataset loaded successfully!")
print("Number of rows:", len(df))
print("Column names:", df.columns.tolist())

df.head()

Dataset loaded successfully!
Number of rows: 8
Column names: ['question', 'answer']


,question,answer
0,How do I reset my password?,Click Forgot Password on the login page and fo...
1,How can I contact support?,Email support@example.com and describe the pro...
2,How do I cancel my subscription?,"Open account settings, select Manage Subscript..."
3,How do I download an invoice?,"Open Billing, find your payment history, and s..."
4,Can I change my email address?,"Open Profile Settings, update your email addre..."


In [4]:
import re

# Remove rows with missing questions or answers
df = df.dropna(subset=["question", "answer"]).copy()

# Clean unnecessary spaces and line breaks
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["question"] = df["question"].apply(clean_text)
df["answer"] = df["answer"].apply(clean_text)

# Remove duplicate questions
df = df.drop_duplicates(subset=["question"])

print("Cleaning complete!")
print("FAQs remaining:", len(df))

df.head()

Cleaning complete!
FAQs remaining: 8


,question,answer
0,How do I reset my password?,Click Forgot Password on the login page and fo...
1,How can I contact support?,Email support@example.com and describe the pro...
2,How do I cancel my subscription?,"Open account settings, select Manage Subscript..."
3,How do I download an invoice?,"Open Billing, find your payment history, and s..."
4,Can I change my email address?,"Open Profile Settings, update your email addre..."


In [5]:
def chunk_text(text, chunk_size=100, overlap=20):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        if end >= len(text):
            break

        start = end - overlap

    return chunks


documents = []
metadata = []

for index, row in df.iterrows():
    text = f"Question: {row['question']} Answer: {row['answer']}"

    chunks = chunk_text(text)

    for chunk_number, chunk in enumerate(chunks):
        documents.append(chunk)

        metadata.append({
            "faq_id": int(index),
            "chunk": chunk_number,
            "question": row["question"]
        })

print("Chunking complete!")
print("Total chunks:", len(documents))

for document in documents[:3]:
    print("\n", document)

Chunking complete!
Total chunks: 16

 Question: How do I reset my password? Answer: Click Forgot Password on the login page and follow the

  page and follow the instructions sent to your email.

 Question: How can I contact support? Answer: Email support@example.com and describe the problem you 


In [9]:
%pip install -q sentence-transformers chromadb

In [10]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# Load a model that converts text into embeddings
embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create a Chroma database
client = chromadb.Client()

# Create a collection to store our FAQ chunks
collection = client.get_or_create_collection(
    name="faq_collection",
    embedding_function=embedding_function
)

# Store the chunks and their metadata
ids = [f"faq_chunk_{i}" for i in range(len(documents))]

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadata
)

print("Embeddings created and stored successfully!")
print("Chunks stored:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created and stored successfully!
Chunks stored: 16


In [11]:
def search_faqs(query):
    results = collection.query(
        query_texts=[query],
        n_results=min(3, collection.count())
    )

    print("Search query:", query)
    print("\nTop 3 matches:\n")

    for i, (document, meta, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):
        print(f"{i}. {meta['question']}")
        print("   Text:", document)
        print(f"   Distance: {distance:.4f}")
        print()

search_faqs("I forgot my login details and cannot access my account")

Search query: I forgot my login details and cannot access my account

Top 3 matches:

1. How do I reset my password?
   Text: Question: How do I reset my password? Answer: Click Forgot Password on the login page and follow the
   Distance: 0.3414

2. How do I delete my account?
   Text: select Delete Account.
   Distance: 0.5340

3. How do I delete my account?
   Text: Question: How do I delete my account? Answer: Open account privacy settings and select Delete Accoun
   Distance: 0.5451



In [12]:
search_faqs("I need a copy of my payment receipt")

Search query: I need a copy of my payment receipt

Top 3 matches:

1. How do I download an invoice?
   Text: Question: How do I download an invoice? Answer: Open Billing, find your payment history, and select 
   Distance: 0.4090

2. How do I download an invoice?
   Text: history, and select Download Invoice.
   Distance: 0.5007

3. Can I change my email address?
   Text: our email address, and verify the new address.
   Distance: 0.7343



In [13]:
search_faqs("I want to stop my monthly plan")

Search query: I want to stop my monthly plan

Top 3 matches:

1. How do I cancel my subscription?
   Text: Question: How do I cancel my subscription? Answer: Open account settings, select Manage Subscription
   Distance: 0.5018

2. How do I cancel my subscription?
   Text:  Manage Subscription, and click Cancel.
   Distance: 0.5148

3. How do I delete my account?
   Text: select Delete Account.
   Distance: 0.6662



In [14]:
search_faqs("I forgot my login details and cannot access my account")
search_faqs("I need a copy of my payment receipt")
search_faqs("I want to stop my monthly plan")

Search query: I forgot my login details and cannot access my account

Top 3 matches:

1. How do I reset my password?
   Text: Question: How do I reset my password? Answer: Click Forgot Password on the login page and follow the
   Distance: 0.3414

2. How do I delete my account?
   Text: select Delete Account.
   Distance: 0.5340

3. How do I delete my account?
   Text: Question: How do I delete my account? Answer: Open account privacy settings and select Delete Accoun
   Distance: 0.5451

Search query: I need a copy of my payment receipt

Top 3 matches:

1. How do I download an invoice?
   Text: Question: How do I download an invoice? Answer: Open Billing, find your payment history, and select 
   Distance: 0.4090

2. How do I download an invoice?
   Text: history, and select Download Invoice.
   Distance: 0.5007

3. Can I change my email address?
   Text: our email address, and verify the new address.
   Distance: 0.7343

Search query: I want to stop my monthly plan

Top 3 matches:

1

In [15]:
%%writefile requirements.txt
pandas
sentence-transformers
chromadb

Writing requirements.txt


In [18]:
%%writefile README.md
# Week 2: Mini Semantic Search

## Overview
This project searches FAQ data by meaning rather than exact keywords.

## Features
- Loads FAQ data using Pandas
- Cleans questions and answers
- Splits text into chunks
- Creates embeddings using Sentence Transformers
- Stores and searches embeddings with ChromaDB
- Prints the top 3 matching FAQ chunks

## Example queries
- I forgot my login details and cannot access my account
- I need a copy of my payment receipt
- I want to stop my monthly plan

## Requirements
Install dependencies with:

pip install -r requirements.txt

## How to run
Open the notebook in Google Colab and run its cells from top to bottom.

## Dataset
The notebook creates a sample FAQ dataset and saves it as faqs.csv.

Overwriting README.md


In [21]:
from google.colab import files

files.download("week-02-mini-semantic-search.ipynb")

FileNotFoundError: Cannot find file: week-02-mini-semantic-search.ipynb